# Exploration of transactions at T3

## Imports

In [1]:
import pandas as pd
import altair as alt

transactions_file = "./priv_clean_data/FACT_Transactions.csv"
payment_methods_file = "./priv_master_data/DIM_Payment_Method.csv"
trucks_file = "./priv_master_data/DIM_Truck.csv"

## Which trucks had the most orders

The chart below shows the total number of orders for each food truck, providing insight into overall customer demand and popularity.

**Kings of Kebabs** leads by a significant margin with 1,322 total orders, demonstrating its strong market position and customer appeal. **Burrito Madness** closely follows with 1,300 orders, and **Cupcakes by Michelle** remains competitive with 1,271 orders, indicating these three trucks maintain consistent, high demand throughout the observation period.

**Yoghurt Heaven** shows moderate performance with 1,052 orders, while the newer entrants **SuperSmoothie** and **Hartmann's Jellied Eels** lag considerably behind with 491 and 291 orders respectively. The significant gap between the top three performers and the lower-volume trucks suggests that product type and market positioning play a major role in customer preferences.

In [2]:
transactions = pd.read_csv(transactions_file)
trucks = pd.read_csv(trucks_file)
df = transactions.merge(trucks, how="left", on="truck_id")

# 1. Count the number of orders per truck
df_orders = df.groupby("truck_name", as_index=False).size().rename(columns={"size": "order_count"})
df_orders["formatted_count"] = df_orders["order_count"].apply(lambda x: f"{x:,}")

# 2. Base chart
base = alt.Chart(df_orders).encode(
    x=alt.X("truck_name:N", title=None, sort="-y",
            axis=alt.Axis(labelAngle=-45))
)

# 3. Layered bars (uses numeric order_count for axis & scale)
bars = base.mark_bar(cornerRadiusTopLeft=6, cornerRadiusTopRight=6).encode(
    y=alt.Y("order_count:Q", title="Number of Orders"),
    color=alt.Color("order_count:Q", scale=alt.Scale(
        scheme="greens"), legend=None),
    tooltip=[
        alt.Tooltip("truck_name:N", title="Truck"),
        alt.Tooltip("formatted_count:N", title="Orders"),
    ],
)

# 4. Text labels using the pre-formatted string
text = base.mark_text(dy=-6, fontWeight="bold").encode(
    y=alt.Y("order_count:Q"),
    text="formatted_count:N",
)

(bars + text).properties(
    title="Number of Orders by Food Truck", width=500, height=300
).configure_view(strokeWidth=0).configure_axis(gridDash=[2, 2], domain=False)

alt.LayerChart(...)

## Which truck was the most profitable

The chart below shows the profitability of each food truck.

We can easily identify that the most profitable food truck is **Kings of Kebabs** followed closely by **Burrito Madness**. However, **Cupcakes by Michelle** lags behind in terms of sales despite having similar popularity to **Burrito Madness** and **Kings of Kebabs**.

Similarly, **Hartmann's Jellied Eels** performs poorly in terms of sales which directly correlates with its lower popularity compared to the other food trucks.

In [3]:
transactions = pd.read_csv(transactions_file)
trucks = pd.read_csv(trucks_file)
df = transactions.merge(trucks, how="left", on="truck_id")

# 1. Sum total spent by truck_name and convert pence to pounds (/ 100)
df_totals = df.groupby("truck_name", as_index=False)["total"].sum()
df_totals["total_pounds"] = df_totals["total"] / 100
df_totals["formatted_total"] = df_totals["total_pounds"].apply(
    lambda x: f"£{x:,.0f}")

# 2. Base chart
base = alt.Chart(df_totals).encode(
    x=alt.X("truck_name:N", title=None, sort="-y",
            axis=alt.Axis(labelAngle=-45))
)

# 3. Layered bars (uses numeric total_pounds for axis & scale)
bars = base.mark_bar(cornerRadiusTopLeft=6, cornerRadiusTopRight=6).encode(
    y=alt.Y("total_pounds:Q", title="Total Spent (£)"),
    color=alt.Color("total_pounds:Q", scale=alt.Scale(
        scheme="greens"), legend=None),
    tooltip=[
        alt.Tooltip("truck_name:N", title="Truck"),
        alt.Tooltip("formatted_total:N", title="Total Spent"),
    ],
)

# 4. Text labels using the pre-formatted string
text = base.mark_text(dy=-6, fontWeight="bold").encode(
    y=alt.Y("total_pounds:Q"),
    text="formatted_total:N",
)

(bars + text).properties(
    title="Total Revenue by Food Truck", width=500, height=300
).configure_view(strokeWidth=0).configure_axis(gridDash=[2, 2], domain=False)

alt.LayerChart(...)

## Order over time

The chart below shows the number of orders for each food truck over the observation period, revealing distinct demand patterns and stability across the business.

We can observe that **Burrito Madness** and **Cupcakes by Michelle** experience significant volatility in daily orders, with peaks reaching over 320 orders but also sharp drops. In contrast, **Kings of Kebabs** demonstrates more consistent and stable order volumes throughout the period, maintaining a steady baseline around 250-300 orders.

**Hartmann's Jellied Eels** consistently remains at the lowest order volume, rarely exceeding 50 orders per day, which aligns with its poor profitability.

In [4]:
transactions = pd.read_csv(transactions_file)
trucks = pd.read_csv(trucks_file)
df = transactions.merge(trucks, how="left", on="truck_id")

# 1. Parse the timestamp and extract the date as a string
df["transaction_date"] = pd.to_datetime(df["at"]).dt.strftime("%Y-%m-%d")

# 2. Count the number of orders per date per truck
df_orders_time = df.groupby(["transaction_date", "truck_name"], as_index=False).size().rename(columns={"size": "order_count"})

# 3. Create line chart with separate line for each truck
chart = alt.Chart(df_orders_time).mark_line(point=True).encode(
    x=alt.X("transaction_date:T", title="Date"),
    y=alt.Y("order_count:Q", title="Number of Orders"),
    color=alt.Color("truck_name:N", title="Truck"),
    tooltip=[
        alt.Tooltip("transaction_date:T", title="Date", format="%Y-%m-%d"),
        alt.Tooltip("truck_name:N", title="Truck"),
        alt.Tooltip("order_count:Q", title="Orders"),
    ]
)

chart.properties(
    title="Orders Over Time by Truck", width=700, height=400
).configure_view(strokeWidth=0).configure_axis(gridDash=[2, 2], domain=False)

alt.Chart(...)